In [1]:
# ==============================================================================
# Cell 1: Setup, NLP, and Punctuation Simulator (Syntactic Space)
# ==============================================================================

"""
Configuración inicial y motor de reconstrucción de texto para el Espacio Sintáctico.
Reutiliza la lógica de puntuación paramétrica basada en la duración de las pausas
para entregar a spaCy un texto con fronteras clausales claras.
"""

import os
import re
from pathlib import Path
from typing import Tuple, List, Dict, Union

import numpy as np
import pandas as pd
import tgt
import spacy

# ------------------------------------------------------------------------------
# Rutas y Modelos
# ------------------------------------------------------------------------------
BASE_DIR = Path("/home/amont21/Documentos/voxelwise modeling/ds003020/derivatives/TextGrids")
SPACY_MODEL_NAME = "en_core_web_trf"

print(f"Cargando modelo de spaCy: {SPACY_MODEL_NAME}...")
nlp = spacy.load(SPACY_MODEL_NAME)

HIGH_RES_FS = 100

# ------------------------------------------------------------------------------
# Parámetros empíricos de puntuación (De la auditoría acústica previa)
# ------------------------------------------------------------------------------
THRESHOLD_PERIOD_SEC = 0.65  
THRESHOLD_COMMA_SEC = 0.35   
NOISE_PATTERN = re.compile(r'\[|\]|\{|\}|\<|\>|spn|^sp$|^sil$|^br$|^lg$', re.IGNORECASE)

def reconstruct_and_map_text(textgrid_path: Union[str, Path]) -> Tuple[str, pd.DataFrame]:
    """Reconstruye el texto con puntuación acústica y mapea los caracteres al tiempo."""
    tg = tgt.io.read_textgrid(str(textgrid_path), include_empty_intervals=True)
    word_tier = next((t for t in tg.get_tier_names() if 'word' in t.lower()), None)
    if not word_tier:
        raise ValueError("Capa 'words' no encontrada.")
        
    raw_tokens, word_mapping = [], []
    
    for interval in tg.get_tier_by_name(word_tier).intervals:
        token = interval.text.strip()
        is_empty = token == ""
        is_noise = bool(NOISE_PATTERN.search(token))
        
        if is_empty or is_noise:
            duration = interval.end_time - interval.start_time
            if duration >= THRESHOLD_PERIOD_SEC:
                if raw_tokens and raw_tokens[-1] not in ['.', ',']:
                    raw_tokens.append('.')
                elif raw_tokens and raw_tokens[-1] == ',':
                    raw_tokens[-1] = '.'
            elif duration >= THRESHOLD_COMMA_SEC:
                if raw_tokens and raw_tokens[-1] not in ['.', ',']:
                    raw_tokens.append(',')
            continue
            
        clean_word = re.sub(r'[^a-zA-Z\']', '', token).lower()
        if clean_word:
            raw_tokens.append(clean_word)
            word_mapping.append({
                'original_word': clean_word,
                'start_time': interval.start_time,
                'end_time': interval.end_time
            })
            
    reconstructed_text = re.sub(r'\s+([.,])', r'\1', " ".join(raw_tokens))
    
    current_idx = 0
    final_mapping = []
    for word_info in word_mapping:
        match = re.search(r'\b' + re.escape(word_info['original_word']) + r'\b', reconstructed_text[current_idx:])
        if match:
            char_start = current_idx + match.start()
            char_end = current_idx + match.end()
            final_mapping.append({**word_info, 'char_start': char_start, 'char_end': char_end})
            current_idx = char_end
            
    return reconstructed_text.strip(), pd.DataFrame(final_mapping)

Cargando modelo de spaCy: en_core_web_trf...


In [2]:
# ==============================================================================
# Cell 2: Syntactic Extraction Engine (Dependency Parsing) - Optimized
# ==============================================================================

"""
Genera la matriz de características sintácticas utilizando árboles de dependencia.

JUSTIFICACIÓN METODOLÓGICA (Parsimonia y Costo de Integración):
Siguiendo las directrices del marco metodológico, se mantiene una parametrización
parsimoniosa de la sintaxis orientada al control del costo de integración clausal:
1. dependency_distance: Controla el costo de retener un elemento en memoria 
   (Working Memory) hasta encontrar su núcleo sintáctico (Gibson, 2000).
2. syntactic_depth: Controla la complejidad por subordinación o anidamiento.
3. is_root: Variable categórica (1 o 0) que controla los picos hemodinámicos 
   asociados a los puntos principales de cierre e integración semántica.
   
NOTA TÉCNICA: Se utiliza np.int16 para optimizar el consumo de memoria RAM de 
la matriz, dado que las distancias y profundidades son conteos enteros.
"""

SYNTACTIC_FEATURE_NAMES = [
    'dependency_distance', 
    'syntactic_depth',     
    'is_root'              
]

def calculate_syntactic_metrics(spacy_token: spacy.tokens.Token) -> list:
    """
    Calcula las métricas estructurales del token en el árbol de dependencias.
    Retorna enteros puros.
    """
    # 1. ¿Es la raíz principal de la oración? (1 o 0)
    is_root = 1 if spacy_token.dep_ == "ROOT" else 0
    
    # 2. Distancia de Dependencia (Absoluta en tokens)
    dep_distance = int(abs(spacy_token.i - spacy_token.head.i))
    
    # 3. Profundidad Estructural (Niveles en el árbol hasta la raíz)
    depth = int(sum(1 for _ in spacy_token.ancestors))
    
    return [dep_distance, depth, is_root]

def extract_syntactic_space(df_alignment: pd.DataFrame, full_text: str, nlp_model: spacy.language.Language, fs: int) -> pd.DataFrame:
    """Proyecta las métricas a 100 Hz."""
    doc = nlp_model(full_text)
    
    char_to_token = {}
    for token in doc:
        for i in range(token.idx, token.idx + len(token.text)):
            char_to_token[i] = token
            
    max_time = df_alignment['end_time'].max() if not df_alignment.empty else 0
    total_samples = int(np.ceil(max_time * fs))
    
    # Optimizamos a int16 (Soporta distancias/profundidades de hasta 32,767)
    feature_matrix = np.zeros((total_samples, len(SYNTACTIC_FEATURE_NAMES)), dtype=np.int16)
    
    for _, row in df_alignment.iterrows():
        mid_char = (row['char_start'] + row['char_end']) // 2
        spacy_token = char_to_token.get(mid_char)
        
        if spacy_token:
            metrics = calculate_syntactic_metrics(spacy_token)
            
            start_idx = int(np.floor(row['start_time'] * fs))
            end_idx = int(np.ceil(row['end_time'] * fs))
            end_idx = min(end_idx, total_samples)
            
            feature_matrix[start_idx:end_idx, :] = metrics
                
    time_axis = np.arange(total_samples) / fs
    df_features = pd.DataFrame(feature_matrix, columns=SYNTACTIC_FEATURE_NAMES, index=time_axis)
    df_features.index.name = 'time_seconds'
    
    return df_features

In [3]:
# ==============================================================================
# Cell 3: Execution and Syntactic Visualization
# ==============================================================================

"""
Ejecuta la extracción sintáctica sobre una historia válida y muestra el 
comportamiento fluctuante de las variables estructurales.
"""

try:
    textgrid_files = list(BASE_DIR.rglob("*.TextGrid"))
    valid_file = next((f for f in textgrid_files if f.name not in ['legacy.TextGrid', 'exorcism.TextGrid']), None)
            
    if valid_file:
        print(f"Extrayendo árbol sintáctico para: {valid_file.name}")
        
        # 1. Reconstruir
        full_text, df_map = reconstruct_and_map_text(valid_file)
        
        # 2. Extraer espacio
        df_syntactic_space = extract_syntactic_space(df_map, full_text, nlp, HIGH_RES_FS)
        
        print("\nInformación del Espacio Sintáctico:")
        print(f"  -> Dimensiones (Muestras x Rasgos): {df_syntactic_space.shape}")
        print(f"  -> Tipo de dato: {df_syntactic_space.values.dtype}\n")
        
        print("Visualización (Nótese cómo varía la distancia y profundidad clausal):")
        
        # Mostrar instantes activos (donde la palabra tiene métricas)
        active_sync_samples = df_syntactic_space[(df_syntactic_space.sum(axis=1) > 0)]
        
        display(active_sync_samples.drop_duplicates().head(15))
        
    else:
        print("Asegúrate de tener un archivo válido seleccionado.")

except Exception as e:
    print(f"Ocurrió un error en la extracción sintáctica: {str(e)}")

Extrayendo árbol sintáctico para: odetostepfather.TextGrid

Información del Espacio Sintáctico:
  -> Dimensiones (Muestras x Rasgos): (73039, 3)
  -> Tipo de dato: int16

Visualización (Nótese cómo varía la distancia y profundidad clausal):


,dependency_distance,syntactic_depth,is_root
time_seconds,,,
0.03,3,1,0
0.22,1,3,0
0.32,2,2,0
0.87,0,0,1
1.00,1,2,0
1.10,2,1,0
1.89,4,1,0
2.00,3,2,0
2.46,8,1,0


In [ ]:
"""
Eso significa que una palabra está a 2 palabras de distancia de su núcleo sintáctico, y está en un nivel 3 de profundidad (muy incrustada en la oración), pero no es la raíz (0.0).
De vez en cuando verás [0.0, 0.0, 1.0]. ¡Esa palabra es la RAÍZ de la oración (el verbo principal)!
"""

In [ ]:
"""
¿Hay algo raro en la salida? ¡Al contrario, es una obra de arte!
Si cruzamos los tiempos de tu tabla con la historia que simulamos antes (odetostepfather: "under the influence is our topic..."), mira lo que acaba de capturar el modelo:
0.03s (under): [Dist: 3, Profundidad: 1, Raíz: 0]
0.22s (the): [Dist: 1, Profundidad: 3, Raíz: 0] (Está muy incrustada porque depende de "influence", que a su vez depende de "under").
0.32s (influence): [Dist: 2, Profundidad: 2, Raíz: 0]
0.87s (is): [Dist: 0, Profundidad: 0, Raíz: 1] 🎯 ¡BINGO! El modelo detectó que el verbo "is" es el ancla principal de la oración (Depth 0, Distancia 0).
1.00s (our): [Dist: 1, Profundidad: 2, Raíz: 0]
1.10s (topic): [Dist: 2, Profundidad: 1, Raíz: 0]
Y fíjate en los tiempos 2.46s (Distancia 8) y 4.71s (Distancia 9). Esto significa que en ese segundo exacto, el oyente escuchó una palabra y su cerebro tuvo que "viajar hacia atrás" 9 palabras en su memoria de trabajo para conectarla sintácticamente con su núcleo. Esta métrica es oro puro para controlar los picos de señal BOLD causados por la memoria de trabajo y la integración sintáctica (la Teoría de Integración de Gibson).
"""